<h2> About </h2>

This notebook retrieves and formats data from RIPE NCC datasets. The data is saved in csv format on local machine and then published in onedrive shared folder.


<h3>Outline</h3>

1. [Setting up the environment](#setting-up-the-environment)
2. [Data extraction](#data-extraction)  
    2.1. [List of country codes](#list-of-country-codes)  
    2.2. [Bandwidth](#bandwidth)  
    2.3. [Country resource list](#country-resource-list)  
    2.4. [Country resource stats](#country-resource-stats)  
3. [Data Merging](#data-merging)



## Setting up the environment

In [1]:
# Import libraries
import requests
import json
import pandas as pd
import os
import re
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
# Set up a path to all the locally stored files
file_path = "C:\\Users\\majak\\OneDrive\\Pulpit\\UvA\\Capstone"

In [3]:
def add_data_to_csv(filepath, newdata):
    """
    Add new data into existing csv file without overwritting previous entries

    filepath (str):
        The path to the csv file
    newdata (DataFrame)
        A DataFrame to append to csv file
    """
    # Check if the path exists
    if os.path.exists(filepath):
        # If exists, write to csv with no headers
        newdata.to_csv(filepath, mode="a", header=False, index=False)
        print(f"Succesfully added data into {filepath}")
    else:
        # If it doesn't, create a new csv
        newdata.to_csv(filepath, mode="w", header=True, index=False)
        print(f"Data has been succesfully saved into a new csv with path {filepath}")

## Data extraction

### List of country codes

In [4]:
# ISO 3166 country codes provided from RIPE
country_list_url = "https://ftp.ripe.net/iso3166-countrycodes.txt"
response = requests.get(country_list_url)

# Check if request was successful and read data
if response.status_code == 200:
    country_list_data = response.text
else:
    print("error while fetching the data")

# Process the text data to retrieve only country codes
country_list_data = [
    re.split(r"\t+|\s{2,}", line) for line in country_list_data.split("\n")
]
country_codes = [
    entry[1] if len(entry) > 1 else entry[0] for entry in country_list_data[11:]
]
# Filter out empty strings and lowercase the codes
country_codes = [code for code in country_codes if code != ""]
country_codes = [code.lower() for code in country_codes]
country_codes = [
    code
    for code in country_codes
    if code != "united kingdom of great britain and northern"
]

### Date range

Specify data range for extrating data

In [5]:
startdate = datetime(2015, 1, 1)
enddate = datetime(2019, 12, 31)

### Bandwidth

In [3]:
"""This part of code is unused as the data is under maintanance"""

# fetch data from the RIPE Atlas API
bandwidth_url = "https://stat.ripe.net/data/mlab-bandwidth/data.json?resource=cy&starttime=2020-08-21&endtime=2020-08-27"
response = requests.get(bandwidth_url)

# check if the request was successful
if response.status_code == 200:
    bandwitdh_data = response.json()
else:
    print("Error fetching data")

# format json for readibility and print
print(json.dumps(bandwitdh_data, indent=4))

{
    "messages": [
        [
            "info",
            "This data is currently unavailable due to maintenance. Please check official announcements for when it will be available again! https://stat.ripe.net/feedback"
        ]
    ],
    "see_also": [],
    "version": "1.0",
    "data_call_name": "mlab-bandwidth",
    "data_call_status": "maintenance - this data call is in maintenance mode",
    "cached": true,
    "data": {},
    "query_id": "20250324120557-eec85d63-4a90-41b2-a48b-9ad9daac3831",
    "process_time": 0,
    "server_id": "app194",
    "build_version": "main-2025.03.24",
    "status": "ok",
    "status_code": 200,
    "time": "2025-03-24T12:05:57.271316"
}


### Country resource list
Raw data source: https://stat.ripe.net/docs/02.data-api/country-resource-list.html

Due to big amount of data this code is run in batches

In [6]:
# Set up retry logic for timedout or failed requests
retries = Retry(
    total=5,  # Retry up to 5 times
    backoff_factor=2,  # Exponential backoff (e.g., 1s, 2s, 4s, 8s, etc.)
    status_forcelist=[500, 502, 503, 504],  # Retry on server-side errors
    method_whitelist=["HEAD", "GET", "OPTIONS"],
)  # Retry on these methods
adapter = HTTPAdapter(max_retries=retries)

# Create a session that will use the retry adapter
session = requests.Session()
session.mount("https://", adapter)


def fetch_data(country, date):
    """General function to retrieve data per request"""
    url = f"https://stat.ripe.net/data/country-resource-list/data.json?resource={country}&time={date}"
    try:
        response = session.get(url, timeout=30)  # Using session with retry & timeout
        response.raise_for_status()  # Raise error for HTTP errors (e.g., 404, 500)

        if response.status_code == 200:
            data = response.json()
            flat_entry = {
                "Country": country,
                "Query_Time": data["data"].get("query_time", ""),
            }

            # Flatten nested 'resources' dictionary
            resources = data.get("data", {}).get("resources", {})
            for key, value in resources.items():
                flat_entry[key] = (
                    ", ".join(map(str, value)) if isinstance(value, list) else value
                )

            all_data.append(flat_entry)
        else:
            print(f"Failed to get data for {country}")
    except requests.exceptions.Timeout:
        print(f"Timeout for {country} on {date}, retrying...")
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {country} on {date}: {e}")
    return None


# Create the list to hold all data
all_data = []

# Use ThreadPoolExecutor to make requests in parallel for each day and country
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = []
    for day in range(0, (enddate - startdate).days + 1):
        current_date = (startdate + timedelta(days=day)).strftime("%Y-%m-%d")
        for country in country_codes:
            futures.append(executor.submit(fetch_data, country, current_date))

    # Collect results from all futures
    for future in futures:
        result = future.result()
        if result:
            all_data.extend(result)

# Save results if data exists
if all_data:
    result_df = pd.DataFrame(all_data)
    add_data_to_csv(os.path.join(file_path, "country_resource_list.csv"), result_df)

else:
    print("No data collected.")

C:\Users\majak\AppData\Local\Temp\ipykernel_12760\493357747.py:2: DeprecationWarning: Using 'method_whitelist' with Retry is deprecated and will be removed in v2.0. Use 'allowed_methods' instead
  retries = Retry(total=5,  # Retry up to 5 times


Succesfully added data into C:\Users\majak\OneDrive\Pulpit\UvA\Capstone\country_resource_list.csv


### Country resource stats
Raw data source: https://stat.ripe.net/docs/02.data-api/country-resource-stats.html

In [ ]:
data_list = []

for country_code in country_codes:
    url = f"https://stat.ripe.net/data/country-resource-stats/data.json?resource={country_code}&starttime=2014-12-31T12:00&endtime=2025-03-17T12:00&resolution=1d"
    response = requests.get(url)
    if response.status_code == 200:
        data_retrieved = response.json()
        data = data_retrieved["data"]["stats"]
        for entry in data:
            timeline_entry = entry["timeline"]
            data_list.append(
                {
                    "statsdate": entry["stats_date"],
                    "starttime": timeline_entry[0].get("starttime"),
                    "endtime": timeline_entry[0].get("endtime"),
                    "v4_prefixes_ris": entry["v4_prefixes_ris"],
                    "v6_prefixes_ris": entry["v6_prefixes_ris"],
                    "asns_ris": entry["asns_ris"],
                    "v4_prefixes_stats": entry["v4_prefixes_stats"],
                    "v6_prefixes_stats": entry["v6_prefixes_stats"],
                    "asns_stats": entry["asns_stats"],
                    "country_code": data_retrieved["data"]["resource"],
                }
            )
    else:
        print(f"Error catching data for {country_code}")

country_stats_df = pd.DataFrame(data_list)
country_stats_df.to_csv(
    "C:\\Users\\majak\\OneDrive\\Pulpit\\UvA\\Capstone\\country_resource_stats.csv"
)

## Data Merging

In [ ]:
# Read the locally saved files
country_list = pd.read_csv(
    "C:\\Users\\majak\\OneDrive\\Pulpit\\UvA\\Capstone\\country_resource_list.csv"
)
stats_data = pd.read_csv(
    "C:\\Users\\majak\\OneDrive\\Pulpit\\UvA\\Capstone\\country_resource_stats.csv"
)

In [ ]:
# Merge the datasets
merged_internet_data = country_list.merge(
    stats_data,
    left_on=["Country", "Query_Time"],
    right_on=["country_code", "statsdate"],
)

# Save to csv
merged_internet_data.to_csv(
    "C:\\Users\\majak\\OneDrive\\Pulpit\\UvA\\Capstone\\internet_data.csv", index=False
)